# Olist · ETL e respostas às três perguntas de negócio

O notebook grava as tabelas regeneradas e os resultados completos em `saida_analise/`. As tabelas da branch em `data/tratados/` servem para conferência.

## 1. Leitura e auditoria das tabelas originais

In [1]:
from pathlib import Path
import pandas as pd

dados = Path("data")
saida = Path("saida_analise")
analytics = dados / "analytics"
analytics.mkdir(exist_ok=True)
chaves = {
    "pedidos": ["order_id"], "clientes": ["customer_id"],
    "produtos": ["product_id"], "itens": ["order_id", "order_item_id"],
    "pagamentos": ["order_id", "payment_sequential"], "avaliacao": ["order_id"]
}
tabelas = {}
auditoria = []
for nome, chave in chaves.items():
    tabela = pd.read_csv(dados / "raw" / f"tabela_{nome}.csv", sep=";")
    tabelas[nome] = tabela
    auditoria.append([nome, len(tabela), tabela.isna().sum().sum(),
                      tabela.duplicated().sum(), tabela.duplicated(chave).sum()])
    assert tabela[chave].notna().all().all(), nome
    if nome != "avaliacao":
        assert not tabela.duplicated(chave).any(), nome
print(pd.DataFrame(auditoria, columns=["tabela", "linhas", "células nulas",
                                      "linhas repetidas", "chaves repetidas"]).to_string(index=False))

    tabela  linhas  células nulas  linhas repetidas  chaves repetidas
   pedidos   99441           2965                 0                 0
  clientes   99441              0                 0                 0
  produtos   32951            618                 0                 0
     itens  112650              0                 0                 0
pagamentos  103886              0                 0                 0
 avaliacao   99224              0               349               551


## 2. Tratamento e conferência

Preservamos avaliações repetidas porque o extrato contém apenas pedido e nota. Frete zero é um valor observado. Pesos zero e parcelas incompatíveis com cartão ficam sinalizados. Datas de entrega ausentes permanecem ausentes. A marca de data de envio identifica a anomalia de 2020 documentada nesta edição da base.

In [2]:
ped = tabelas["pedidos"]
cli = tabelas["clientes"]
prod = tabelas["produtos"]
itens = tabelas["itens"]
pag = tabelas["pagamentos"]
av = tabelas["avaliacao"]

for coluna in ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    ped[coluna] = pd.to_datetime(ped[coluna], errors="raise")
entregue = ped.order_status.eq("delivered")
ped["entregue_sem_data_entrega"] = entregue & ped.order_delivered_customer_date.isna()
ped["nao_entregue_com_data_entrega"] = ~entregue & ped.order_delivered_customer_date.notna()

cli["customer_city"] = cli.customer_city.str.strip().str.lower()
cli["customer_state"] = cli.customer_state.str.strip().str.upper()
prod["product_category_name"] = prod.product_category_name.fillna("nao_definido")
prod["peso_inconsistente"] = prod.product_weight_g.eq(0)
prod.loc[prod.peso_inconsistente, "product_weight_g"] = float("nan")
itens["shipping_limit_date"] = pd.to_datetime(itens.shipping_limit_date)
assert itens.price.gt(0).all() and itens.freight_value.ge(0).all()
itens["total_value"] = itens.price + itens.freight_value
itens["frete_zero"] = itens.freight_value.eq(0)
itens["data_envio_inconsistente"] = itens.shipping_limit_date.dt.year.eq(2020)

pag["pagamento_valor_zero"] = pag.payment_value.eq(0)
pag["parcelas_inconsistentes"] = pag.payment_type.eq("credit_card") & pag.payment_installments.eq(0)
pag["payment_installments"] = pag.payment_installments.astype("Int64").mask(pag.parcelas_inconsistentes)
pag["payment_type"] = pag.payment_type.replace("not_defined", pd.NA)
assert av.review_score.isin(range(1, 6)).all()
av["pedido_com_multiplas_avaliacoes"] = av.duplicated("order_id", keep=False)
av["pedido_nota_repetidos"] = av.duplicated(["order_id", "review_score"], keep=False)

tratados = saida / "tratados"
tratados.mkdir(parents=True, exist_ok=True)
for nome, tabela in tabelas.items():
    arquivo = f"tabela_{nome}.csv"
    tabela.to_csv(tratados / arquivo, sep=";", index=False)
    esperado = pd.read_csv(dados / "cleansed" / arquivo, sep=";")
    obtido = pd.read_csv(tratados / arquivo, sep=";")
    pd.testing.assert_frame_equal(obtido, esperado)
print("As seis tabelas regeneradas conferem com os CSVs tratados.")

As seis tabelas regeneradas conferem com os CSVs tratados.


## 3. População e unidades de análise

Usamos pedidos entregues com datas real e prevista conhecidas nas três perguntas. A unidade de entrega é o pedido. A nota do pedido é a média de suas avaliações disponíveis. Para a análise por categoria, um pedido conta uma vez em cada categoria comprada. Os totais por categoria podem, portanto, superar o número global de pedidos. A nota descreve a experiência com o pedido inteiro. Valor de mercadorias é a soma dos preços dos itens, em reais, sem frete.

In [3]:
elegivel = entregue & ped.order_delivered_customer_date.notna() & ped.order_estimated_delivery_date.notna()
pedidos = ped.loc[elegivel].merge(cli[["customer_id", "customer_state"]], how="left",
                                  on="customer_id", validate="many_to_one")
assert pedidos.customer_state.notna().all()
pedidos["atrasado"] = (pedidos.order_delivered_customer_date.dt.normalize()
                       > pedidos.order_estimated_delivery_date.dt.normalize())
notas = av.groupby("order_id").review_score.mean().rename("nota")
pedidos = pedidos.join(notas, on="order_id", validate="one_to_one")
vendas = itens.merge(pedidos[["order_id", "atrasado", "nota"]], on="order_id", validate="many_to_one")
vendas = vendas.merge(prod, on="product_id", how="left", validate="many_to_one")
assert vendas.product_category_name.notna().all()
cat = "product_category_name"
pc = vendas[["order_id", cat, "atrasado", "nota"]].drop_duplicates(["order_id", cat])
print("Pedidos originais / elegíveis / avaliados / com itens:",
      len(ped), len(pedidos), pedidos.nota.count(), vendas.order_id.nunique())
print(pedidos.groupby("atrasado").nota.agg(["size", "count", "mean"]))

Pedidos originais / elegíveis / avaliados / com itens: 99441 96470 95824 96470
           size  count      mean
atrasado                        
False     89936  89443  4.290589
True       6534   6381  2.271823


## Pergunta 1 · Onde o atraso está associado a menor satisfação?

Diferença em pontos = nota média no prazo menos nota média em atraso. Cada média divide a soma das notas dos pedidos avaliados pelo número de pedidos avaliados naquele grupo. Ordenamos grupos com pelo menos 30 pedidos avaliados em cada condição para reduzir a influência de amostras muito pequenas. O CSV também mostra os demais grupos e os pedidos sem nota entram na contagem de cobertura.

In [4]:
def comparar_notas(dados_grupo, chave):
    resumo = dados_grupo.groupby([chave, "atrasado"]).nota.agg(
        pedidos="size", avaliados="count", media="mean").unstack("atrasado")
    resumo.columns = [f"{medida}_{'atraso' if atraso else 'prazo'}" for medida, atraso in resumo.columns]
    resumo["diferenca_pontos"] = resumo.media_prazo - resumo.media_atraso
    resumo["base_suficiente"] = (resumo.avaliados_prazo.ge(30) & resumo.avaliados_atraso.ge(30))
    return resumo.sort_values("diferenca_pontos", ascending=False)

for chave, base in [("customer_state", pedidos), (cat, pc)]:
    resumo = comparar_notas(base, chave)
    resumo.to_csv(analytics / f"pergunta_1_{chave}.csv")
    print(resumo[resumo.base_suficiente].head(5).round(2).to_string())

                pedidos_prazo  pedidos_atraso  avaliados_prazo  avaliados_atraso  media_prazo  media_atraso  diferenca_pontos  base_suficiente
customer_state                                                                                                                                
RN                        430              44              428                43         4.39          1.77              2.62             True
RJ                      10855            1495            10755              1456         4.24          1.91              2.33             True
SE                        284              51              284                50         4.25          1.94              2.31             True
DF                       1962             118             1952               118         4.26          1.97              2.29             True
PE                       1440             153             1430               149         4.30          2.03              2.26             True

## Pergunta 2 · Onde o frete pesa mais e como se relaciona com as medidas físicas?

Frete proporcional da categoria = 100 × soma dos fretes dos itens / soma dos preços dos mesmos itens. Incluímos itens com frete zero. Exibimos categorias identificadas com pelo menos 100 pedidos elegíveis. Para a associação física, calculamos a correlação de Spearman por item vendido, usando os pares com medidas disponíveis. Comparamos frete em reais e frete/preço para distinguir as duas relações. São associações descritivas, sujeitas a diferenças de preço, rota e categoria.

In [5]:
comercial = vendas.groupby(cat).agg(itens=("order_id", "size"), pedidos=("order_id", "nunique"),
                                    valor_mercadorias=("price", "sum"), frete=("freight_value", "sum"))
comercial["frete_pct"] = 100 * comercial.frete / comercial.valor_mercadorias
comercial.sort_values("frete_pct", ascending=False).to_csv(analytics / "pergunta_2_categorias.csv")
identificadas = comercial[comercial.index != "nao_definido"]
print(identificadas[identificadas.pedidos.ge(100)].nlargest(5, "frete_pct").round(2).to_string())

vendas["volume_cm3"] = vendas.product_length_cm * vendas.product_width_cm * vendas.product_height_cm
vendas["frete_preco"] = vendas.freight_value / vendas.price
associacoes = []
for medida in ["product_weight_g", "volume_cm3", "product_length_cm", "product_width_cm", "product_height_cm"]:
    pares = vendas[[medida, "freight_value", "frete_preco"]].dropna()
    ranks = pares.rank()
    associacoes.append([medida, len(pares), ranks[medida].corr(ranks.freight_value),
                        ranks[medida].corr(ranks.frete_preco)])
fisico = pd.DataFrame(associacoes, columns=["medida", "itens_validos", "rho_frete_reais", "rho_frete_preco"])
print(fisico.round(3).to_string(index=False))
fisico.to_csv(analytics / "pergunta_2_medidas.csv", index=False)

                         itens  pedidos  valor_mercadorias     frete  frete_pct
product_category_name                                                          
artigos_de_natal           150      125            8737.84   3190.91      36.52
sinalizacao_e_seguranca    197      138           21315.05   6477.54      30.39
eletronicos               2729     2517          155043.93  45679.16      29.46
alimentos_bebidas          269      221           14942.88   4394.89      29.41
moveis_sala                495      414           67350.70  17694.37      26.27
           medida  itens_validos  rho_frete_reais  rho_frete_preco
 product_weight_g         110163            0.446           -0.271
       volume_cm3         110171            0.368           -0.180
product_length_cm         110171            0.283           -0.101
 product_width_cm         110171            0.273           -0.117
product_height_cm         110171            0.282           -0.166


## Pergunta 3 · Quais categorias relevantes são críticas nos três indicadores?

Definimos relevância como as dez categorias identificadas com maior valor de mercadorias no recorte. Uma categoria é crítica quando, simultaneamente, sua taxa de atraso e seu frete proporcional superam as referências globais e sua nota média fica abaixo da média global. Taxa de atraso divide pedidos atrasados pelos pedidos elegíveis do grupo. A nota usa apenas os pedidos avaliados. As referências globais contam cada pedido uma vez e incluem todas as categorias no cálculo monetário.

In [6]:
referencia = pd.Series({
    "atraso_pct": 100 * pedidos.atrasado.mean(),
    "frete_pct": 100 * vendas.freight_value.sum() / vendas.price.sum(),
    "nota_media": pedidos.nota.mean()
})
operacao = pc.groupby(cat).agg(atrasados=("atrasado", "sum"), avaliados=("nota", "count"),
                               nota_media=("nota", "mean"))
ranking = identificadas.nlargest(10, "valor_mercadorias").join(operacao)
ranking["atraso_pct"] = 100 * ranking.atrasados / ranking.pedidos
ranking["critica"] = (ranking.atraso_pct.gt(referencia.atraso_pct)
                      & ranking.frete_pct.gt(referencia.frete_pct)
                      & ranking.nota_media.lt(referencia.nota_media))
print("Referências globais:", referencia.round(3).to_dict())
print(ranking.round(2).to_string())
ranking.to_csv(analytics / "pergunta_3_top10.csv")

Referências globais: {'atraso_pct': 6.773, 'frete_pct': 16.627, 'nota_media': 4.156}
                        itens  pedidos  valor_mercadorias      frete  frete_pct  atrasados  avaliados  nota_media  atraso_pct  critica
product_category_name                                                                                                                 
beleza_saude             9465     8647         1233131.72  178957.81      14.51        649       8601        4.23        7.51    False
relogios_presentes       5857     5493         1165898.98   98118.00       8.42        406       5452        4.12        7.39    False
cama_mesa_banho         10953     9272         1023434.76  201774.50      19.72        689       9177        4.00        7.43     True
esporte_lazer            8430     7529          954673.55  163389.36      17.11        495       7485        4.23        6.57    False
informatica_acessorios   7643     6529          888613.62  143990.03      16.20        417       6498    

## Respostas

1. **Atraso e satisfação.** Entre grupos com pelo menos 30 avaliações em cada condição, as maiores quedas de nota aparecem no RN (2,62 pontos), RJ (2,33) e SE (2,31). Nas categorias, destacam-se instrumentos musicais (2,51), áudio (2,39) e brinquedos (2,24). O RJ reúne 1.456 pedidos atrasados avaliados, contra 43 no RN. A diferença observada é uma associação e o tamanho das amostras importa para interpretar a ordem.

2. **Frete.** Entre categorias com pelo menos 100 pedidos, artigos de Natal lidera (36,52%), seguida de sinalização e segurança (30,39%) e eletrônicos (29,46%). Peso e volume têm associação positiva com o frete em reais e negativa com a proporção frete/preço neste conjunto de itens. O preço do produto entra no denominador, e as correlações agregadas não isolam um efeito de peso ou volume.

3. **Prioridade comercial.** Entre as dez categorias de maior valor vendido, cama, mesa e banho e móveis e decoração atendem aos três critérios simultaneamente. A primeira tem maior valor vendido entre as duas e 689 pedidos atrasados. Na segunda, o frete equivale a 23,65% do valor das mercadorias, ante 16,63% no conjunto.